# Signals and Filters: the $z$-Transform, Poles and Zeros

A digital filter is a short recipe: each output sample is a weighted sum of recent inputs and recent outputs. Such **difference equations** run inside every audio plug-in, sensor, modem and control loop. The $z$-transform packs a whole signal into one function of a complex variable, turns the recipe into a rational function $H(z)$, and hands its behaviour to geometry: the **poles** of $H$ decide stability (inside the unit circle or not), and the poles and zeros together shape the **frequency response** on the unit circle.

This notebook defines the $z$-transform as a Laurent series, derives transfer functions and impulse responses, proves the unit-circle stability test, reads frequency responses off pole–zero plots, and designs a notch filter that removes 50 Hz mains hum — implementing the recurrence directly, with no signal-processing library.

This is notebook 04 of module 07 (**Applications**) in the **Learning Complex Numbers** series.

## 1. Introduction

Notebook 02 solved continuous-time systems with the ansatz $e^{\lambda t}$ and read stability off the left half-plane. Notebook 03 sampled signals and analysed them with the DFT on the unit circle. This notebook joins the two threads for **discrete-time systems**:

| Continuous time | Discrete time |
|---|---|
| differential equation | difference equation |
| $e^{\lambda t}$, with $\frac{d}{dt} \mapsto \lambda$ | $z^n$, with delay $\mapsto z^{-1}$ |
| Laplace transform, $s$-plane | $z$-transform, $z$-plane |
| stable $\iff$ poles in $\mathrm{Re}\,s < 0$ | stable $\iff$ poles in $\lvert z\rvert < 1$ |
| frequency response on the imaginary axis $s = i\omega$ | frequency response on the unit circle $z = e^{i\omega}$ |

The dictionary between the two columns is $z = e^{sT}$ with sampling interval $T$: it maps the left half-plane to the inside of the unit disk and the imaginary axis onto the unit circle.

**Prerequisites**

- Power series, geometric series and convergence — [Power series and radius of convergence](../06_complex_integration/04_power_series_and_radius_of_convergence.ipynb).
- Laurent series and poles — [Laurent series and singularities](../06_complex_integration/05_laurent_series_and_singularities.ipynb).
- Partial fractions over $\mathbb{C}$ — [Partial fractions over C](../02_polynomials_and_roots/04_partial_fractions_over_c.ipynb).
- Characteristic equations and stability — [Linear ODEs and the damped oscillator](02_linear_odes_and_the_damped_oscillator.ipynb).
- Sampling, the DFT and convolution — [Fourier series and the DFT](03_fourier_series_and_the_dft.ipynb).

After completing this notebook you will be able to:

- Compute $z$-transforms of simple sequences and state their region of convergence.
- Turn a difference equation into a transfer function $H(z)$ and find its poles and zeros.
- Decide stability from the pole positions and predict the shape of the impulse response.
- Evaluate and sketch the frequency response $H(e^{i\omega})$ geometrically from a pole–zero plot.
- Design a simple notch or resonator filter by placing poles and zeros, and run it on a signal.

## 2. Theory

### 2.1 Difference equations

A causal **linear time-invariant** (LTI) filter with input $x[n]$ and output $y[n]$ is

$$
y[n] + a_1 y[n-1] + \dots + a_P\,y[n-P] = b_0 x[n] + b_1 x[n-1] + \dots + b_Q\,x[n-Q].
$$

Solved for $y[n]$ it is a recipe: $y[n] = \sum_k b_k x[n-k] - \sum_{k \ge 1} a_k y[n-k]$. If all $a_k = 0$ the filter has a finite impulse response (**FIR**, e.g. a moving average); otherwise feedback makes it **IIR** (infinite impulse response). The **impulse response** $h[n]$ is the output for the input $\delta[n] = (1, 0, 0, \dots)$, and by linearity the output for any input is the convolution $y = h * x$.

### 2.2 The $z$-transform

For a sequence $x[n]$ with $x[n] = 0$ for $n < 0$, define

$$
X(z) = \sum_{n=0}^{\infty} x[n]\,z^{-n}.
$$

This is a **Laurent series** in $z$ (a power series in $w = 1/z$). It converges outside some circle, $|z| > R$: the **region of convergence** (ROC). The basic example is the geometric sequence:

$$
x[n] = a^n \quad\Longrightarrow\quad X(z) = \sum_{n \ge 0}\left(\frac{a}{z}\right)^n = \frac{1}{1 - az^{-1}} = \frac{z}{z - a}, \qquad |z| > |a|.
$$

It has a **pole** at $z = a$: the base of the geometric sequence. Two properties do almost all the work:

- **Linearity:** $\alpha x + \beta y \mapsto \alpha X + \beta Y$.
- **Delay:** $x[n - 1] \mapsto z^{-1}X(z)$ (with $x[-1] = 0$), since $\sum_n x[n-1]z^{-n} = z^{-1}\sum_m x[m]z^{-m}$. So $z^{-1}$ is the "one-sample delay" operator.

Consequently convolution becomes multiplication, $h * x \mapsto H(z)X(z)$ — the same theorem as in notebook 03.

### 2.3 Transfer function, poles and zeros

Transforming the difference equation term by term:

$$
H(z) = \frac{Y(z)}{X(z)} = \frac{b_0 + b_1 z^{-1} + \dots + b_Q z^{-Q}}{1 + a_1 z^{-1} + \dots + a_P z^{-P}}.
$$

Multiplying top and bottom by a power of $z$ shows that $H$ is a ratio of polynomials in $z$:

- **zeros** = roots of $b_0 z^Q + b_1 z^{Q-1} + \dots + b_Q$ (computed by `np.roots(b)`),
- **poles** = roots of $z^P + a_1 z^{P-1} + \dots + a_P$ (computed by `np.roots(a)`),

plus extra poles or zeros at $z = 0$ if $P \ne Q$, which are pure delays. Real coefficients mean poles and zeros come in **conjugate pairs**.

### 2.4 Impulse response and stability

If the poles $p_1, \dots, p_P$ are distinct, partial fractions give $H(z) = c + \sum_j \frac{r_j}{1 - p_j z^{-1}}$, and inverting each geometric term,

$$
h[n] = \sum_j r_j\,p_j^{\,n} \qquad (n \ge 1).
$$

Each pole contributes a geometric sequence. Writing $p = \rho e^{i\theta}$, a conjugate pair contributes $2|r|\rho^n\cos(\theta n + \arg r)$: a **damped (or growing) oscillation** with ratio $\rho$ per sample and $\theta$ radians per sample.

A filter is **BIBO stable** (every bounded input gives a bounded output) exactly when $\sum_n |h[n]| < \infty$. For a causal rational filter this holds if and only if

$$
\boxed{\text{every pole satisfies } |p_j| < 1.}
$$

Poles inside the unit circle: $h$ decays geometrically. On the circle: $h$ oscillates forever (marginal). Outside: $h$ blows up.

### 2.5 Frequency response: evaluate on the unit circle

Feed in a complex sinusoid $x[n] = e^{i\omega n}$, i.e. $z_0^n$ with $z_0 = e^{i\omega}$. Then

$$
y[n] = \sum_k h[k]\,e^{i\omega(n - k)} = \underbrace{\left(\sum_k h[k]e^{-i\omega k}\right)}_{H(e^{i\omega})} e^{i\omega n}.
$$

Sinusoids are eigenfunctions: the filter multiplies each one by the complex number $H(e^{i\omega})$, whose modulus is the **gain** and whose argument is the **phase shift** at frequency $\omega \in [0, \pi]$ radians per sample ($\omega = \pi$ is the Nyquist frequency $f_s/2$). Sampling $\omega = 2\pi k/N$ gives the DFT of $h$ (zero-padded to length $N$).

**Geometric reading.** In factored form $H(z) = b_0\,z^{P-Q}\,\dfrac{\prod_k (z - q_k)}{\prod_j (z - p_j)}$, and $|z^{P-Q}| = 1$ on the circle, so

$$
\left|H(e^{i\omega})\right| = |b_0|\,\frac{\prod_k \left|e^{i\omega} - q_k\right|}{\prod_j \left|e^{i\omega} - p_j\right|}.
$$

Walk around the upper half of the unit circle from $\omega = 0$ (the point $1$) to $\omega = \pi$ (the point $-1$). The gain is a product of distances to zeros divided by a product of distances to poles:

- passing **close to a pole** makes the gain large (a resonant peak),
- passing **close to (or through) a zero** makes it small (a notch).

### 2.6 Designing by placing poles and zeros

- **Notch at $\omega_0$:** zeros *on* the circle at $e^{\pm i\omega_0}$ kill that frequency exactly; poles just inside at $re^{\pm i\omega_0}$ ($r$ slightly below $1$) make the notch narrow, because away from $\omega_0$ each pole nearly cancels the distance factor of its zero:
$$
H(z) = g\,\frac{1 - 2\cos\omega_0\,z^{-1} + z^{-2}}{1 - 2r\cos\omega_0\,z^{-1} + r^2 z^{-2}}.
$$
- **Resonator (band-pass) at $\omega_0$:** poles at $re^{\pm i\omega_0}$, zeros at $\pm1$ to suppress DC and Nyquist:
$$
H(z) = g\,\frac{1 - z^{-2}}{1 - 2r\cos\omega_0\,z^{-1} + r^2 z^{-2}}.
$$

The closer $r$ is to $1$, the sharper the peak or notch (bandwidth roughly $2(1 - r)$ radians per sample) — and the longer the impulse response rings, since $h \sim r^n$.

## 3. Worked Examples

### Example 1 — a one-pole low-pass: $y[n] = 0.5\,y[n-1] + x[n]$

1. In standard form $y[n] - 0.5\,y[n-1] = x[n]$, so $a = (1, -0.5)$, $b = (1)$ and $H(z) = \dfrac{1}{1 - 0.5z^{-1}} = \dfrac{z}{z - 0.5}$.
2. Pole $0.5$, zero $0$. The pole is inside the unit circle: stable. $h[n] = 0.5^n$.
3. Gain at DC ($z = 1$): $\frac{1}{0.5} = 2$. Gain at Nyquist ($z = -1$): $\frac{1}{1.5} = 0.667$. The pole sits near $\omega = 0$, so low frequencies are boosted: a low-pass filter.

### Example 2 — an unstable recursion: $y[n] = 2\,y[n-1] + x[n]$

$H(z) = \frac{1}{1 - 2z^{-1}}$ has a pole at $2$, outside the unit circle; $h[n] = 2^n$. A single input spike makes the output explode.

### Example 3 — a pair of complex poles

$y[n] = 2r\cos\theta\,y[n-1] - r^2y[n-2] + x[n]$ with $r = 0.9$, $\theta = \pi/4$:

1. $H(z) = \dfrac{1}{1 - 2r\cos\theta\,z^{-1} + r^2z^{-2}} = \dfrac{1}{(1 - pz^{-1})(1 - \bar pz^{-1})}$ with $p = 0.9e^{i\pi/4}$.
2. Partial fractions: $\dfrac{1}{(1 - pz^{-1})(1 - \bar pz^{-1})} = \dfrac{1}{p - \bar p}\left(\dfrac{p}{1 - pz^{-1}} - \dfrac{\bar p}{1 - \bar pz^{-1}}\right)$.
3. Inverting: $h[n] = \dfrac{p^{n+1} - \bar p^{\,n+1}}{p - \bar p} = \dfrac{2i\,r^{n+1}\sin((n+1)\theta)}{2i\,r\sin\theta} = r^n\,\dfrac{\sin((n+1)\theta)}{\sin\theta}$.

A damped oscillation: shrinks by $0.9$ per sample and turns $45°$ per sample (period $8$ samples).

### Example 4 — a two-point moving average

$y[n] = \frac12(x[n] + x[n-1])$: $H(z) = \frac12(1 + z^{-1}) = \dfrac{z + 1}{2z}$, a zero at $-1$ and a pole at $0$. On the circle,

$$
H(e^{i\omega}) = \tfrac12\left(1 + e^{-i\omega}\right) = e^{-i\omega/2}\cos\tfrac{\omega}{2}, \qquad |H| = \cos\tfrac{\omega}{2}.
$$

Gain $1$ at DC, $0$ at Nyquist (the zero sits there), and a linear phase $-\omega/2$: a half-sample delay.

### Example 5 — transform of a sampled cosine

$\cos(\theta n) = \frac12\left(e^{i\theta n} + e^{-i\theta n}\right)$, so by linearity and the geometric formula

$$
\mathcal{Z}\{\cos\theta n\} = \frac12\left(\frac{1}{1 - e^{i\theta}z^{-1}} + \frac{1}{1 - e^{-i\theta}z^{-1}}\right) = \frac{1 - \cos\theta\,z^{-1}}{1 - 2\cos\theta\,z^{-1} + z^{-2}}, \quad |z| > 1.
$$

Poles at $e^{\pm i\theta}$, exactly on the unit circle: an undamped oscillation.

## 4. Python Examples

### 4.1 The filter is just the recurrence

`run_filter(b, a, x)` implements section 2.1 literally, with $a_0 = 1$. (This is what `scipy.signal.lfilter` does, but we do not need SciPy.)

In [ ]:
import numpy as np


def run_filter(b, a, x):
    # y[n] = sum_k b[k] x[n-k] - sum_{k>=1} a[k] y[n-k], with a[0] = 1
    b = np.asarray(b, dtype=float)
    a = np.asarray(a, dtype=float)
    assert abs(a[0] - 1) < 1e-12, "normalise so that a[0] = 1"
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    for n in range(len(x)):
        total = 0.0
        for k in range(len(b)):
            if n - k >= 0:
                total += b[k] * x[n - k]
        for k in range(1, len(a)):
            if n - k >= 0:
                total -= a[k] * y[n - k]
        y[n] = total
    return y


np.set_printoptions(precision=5, suppress=True)
impulse = np.zeros(12)
impulse[0] = 1.0
print("Ex 1 h[n]:", np.round(run_filter([1], [1, -0.5], impulse), 5))
print("    0.5^n:", np.round(0.5 ** np.arange(12), 5))
print("Ex 2 h[n]:", run_filter([1], [1, -2], impulse))

r, theta = 0.9, np.pi / 4
h3 = run_filter([1], [1, -2 * r * np.cos(theta), r**2], impulse)
formula = r ** np.arange(12) * np.sin((np.arange(12) + 1) * theta) / np.sin(theta)
print("Ex 3 matches r^n sin((n+1)theta)/sin(theta):", np.allclose(h3, formula))

### 4.2 Poles, zeros and the stability test

In [ ]:
import numpy as np


def poles_and_zeros(b, a):
    zeros = np.roots(b) if len(b) > 1 else np.array([])
    poles = np.roots(a) if len(a) > 1 else np.array([])
    return zeros, poles


systems = {
    "Ex 1  y = 0.5 y[n-1] + x": ([1], [1, -0.5]),
    "Ex 2  y = 2 y[n-1] + x": ([1], [1, -2]),
    "Ex 3  complex pair": ([1], [1, -2 * 0.9 * np.cos(np.pi / 4), 0.81]),
    "Ex 4  moving average": ([0.5, 0.5], [1]),
    "Ex 5  cosine generator": ([1, -np.cos(1.0)], [1, -2 * np.cos(1.0), 1]),
}
for name, (b, a) in systems.items():
    zeros, poles = poles_and_zeros(b, a)
    largest = np.max(np.abs(poles)) if len(poles) else 0.0
    verdict = "stable" if largest < 1 - 1e-12 else ("marginal" if abs(largest - 1) < 1e-12 else "unstable")
    print(f"{name:26s} zeros {np.round(zeros, 4)}  poles {np.round(poles, 4)}  max|p| = {largest:.3f} -> {verdict}")

### 4.3 Symbolic checks: a geometric series and a partial-fraction inverse

The poles in $w = 1/z$ are at $w = 1/p$ and $1/\bar p$; expanding $H$ as a power series in $w$ lists $h[0], h[1], \dots$ as its coefficients.

In [ ]:
import sympy as sp

n = sp.symbols("n", integer=True, nonnegative=True)
z, a = sp.symbols("z a")
print("sum a^n z^-n =", sp.simplify(sp.summation((a / z) ** n, (n, 0, sp.oo))))

# Example 3 with exact numbers. In w = 1/z the denominator is (1 - p w)(1 - conj(p) w)
# = 1 - 2 Re(p) w + |p|^2 w^2 with p = 0.9 e^{i pi/4}, so 2 Re(p) = 0.9 sqrt(2) and |p|^2 = 0.81.
w = sp.symbols("w")
H = 1 / (1 - sp.Rational(9, 10) * sp.sqrt(2) * w + sp.Rational(81, 100) * w**2)
print("partial fractions in w:", sp.apart(H, w, full=True).doit())

h_formula = sp.Rational(9, 10) ** n * sp.sin((n + 1) * sp.pi / 4) / sp.sin(sp.pi / 4)
series = sp.series(H, w, 0, 6).removeO()
print("series coefficients:", [sp.nsimplify(series.coeff(w, k)) for k in range(6)])
print("formula h[n]        :", [sp.nsimplify(h_formula.subs(n, k)) for k in range(6)])

### 4.4 Frequency response, and its link to the DFT

`frequency_response(b, a, omega)` evaluates $H(e^{i\omega})$ directly from the coefficients. For an FIR filter it agrees with the DFT of the impulse response at $\omega = 2\pi k/N$.

In [ ]:
import numpy as np


def frequency_response(b, a, omega):
    e = np.exp(-1j * np.asarray(omega))
    numerator = sum(bk * e**k for k, bk in enumerate(b))
    denominator = sum(ak * e**k for k, ak in enumerate(a))
    return numerator / denominator


print("Ex 1 gain at DC and Nyquist:", np.abs(frequency_response([1], [1, -0.5], [0, np.pi])))
omega = np.array([0, np.pi / 3, np.pi / 2, np.pi])
print("Ex 4 |H|:", np.round(np.abs(frequency_response([0.5, 0.5], [1], omega)), 6), " cos(omega/2):", np.round(np.cos(omega / 2), 6))

N = 16
h = np.ones(5) / 5
dft = np.fft.fft(h, N)
on_circle = frequency_response(h, [1], 2 * np.pi * np.arange(N) / N)
print("DFT of h (zero-padded) == H on the circle:", np.allclose(dft, on_circle))

### 4.5 Designing a notch filter to remove 50 Hz hum

A slow "measurement" (components at $3$ Hz and $11$ Hz) sampled at $f_s = 1000$ Hz is contaminated by mains hum at $50$ Hz with amplitude $1$. The hum sits at $\omega_0 = 2\pi\cdot 50/1000 = \pi/10$ radians per sample. Place zeros at $e^{\pm i\pi/10}$ and poles at $0.95e^{\pm i\pi/10}$, and scale by $g$ so that the DC gain is exactly $1$.

In [ ]:
import numpy as np


def run_filter(b, a, x):
    y = np.zeros(len(x))
    for n in range(len(x)):
        total = sum(b[k] * x[n - k] for k in range(len(b)) if n - k >= 0)
        total -= sum(a[k] * y[n - k] for k in range(1, len(a)) if n - k >= 0)
        y[n] = total
    return y


def frequency_response(b, a, omega):
    e = np.exp(-1j * np.asarray(omega))
    return sum(bk * e**k for k, bk in enumerate(b)) / sum(ak * e**k for k, ak in enumerate(a))


sample_rate, hum_frequency, r = 1000, 50, 0.95
omega_0 = 2 * np.pi * hum_frequency / sample_rate
b = np.array([1, -2 * np.cos(omega_0), 1])
a = np.array([1, -2 * r * np.cos(omega_0), r**2])
b = b * (a.sum() / b.sum())           # g = A(1)/B(1) makes H(1) = 1
print("b =", np.round(b, 5), " a =", np.round(a, 5))
print("zeros:", np.round(np.roots(b), 4), " |zeros| =", np.round(np.abs(np.roots(b)), 4))
print("poles:", np.round(np.roots(a), 4), " |poles| =", np.round(np.abs(np.roots(a)), 4))
for f in [0, 3, 11, 45, 50, 55, 200]:
    print(f"  gain at {f:3d} Hz: {abs(frequency_response(b, a, 2 * np.pi * f / sample_rate)):.4f}")

t = np.arange(0, 2.0, 1 / sample_rate)
measurement = np.sin(2 * np.pi * 3 * t) + 0.4 * np.sin(2 * np.pi * 11 * t)
recorded = measurement + 1.0 * np.sin(2 * np.pi * hum_frequency * t + 0.3)
cleaned = run_filter(b, a, recorded)

second_half = t >= 1.0
error_before = np.sqrt(np.mean((recorded - measurement)[second_half] ** 2))
error_after = np.sqrt(np.mean((cleaned - measurement)[second_half] ** 2))
print(f"RMS deviation from the true measurement (t >= 1 s): before {error_before:.4f}, after {error_after:.4f}")

The residual after filtering is not zero because a notch of finite width also slightly changes the gain and phase at $3$ and $11$ Hz; it is far smaller than the hum it removed.

## 5. Visualizations

### 5.1 Pole position decides the impulse response

Top row: pole–zero plots (× poles, ○ zeros, unit circle dashed). Bottom row: the corresponding impulse responses. Moving a pole outward changes decay into sustained oscillation into blow-up; moving it around the circle changes the oscillation frequency.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cases = [
    ("real pole 0.8", [0.8]),
    ("real pole -0.8", [-0.8]),
    ("$0.9 e^{\\pm i\\pi/6}$", [0.9 * np.exp(1j * np.pi / 6), 0.9 * np.exp(-1j * np.pi / 6)]),
    ("$e^{\\pm i\\pi/6}$ (on circle)", [np.exp(1j * np.pi / 6), np.exp(-1j * np.pi / 6)]),
    ("$1.05 e^{\\pm i\\pi/6}$", [1.05 * np.exp(1j * np.pi / 6), 1.05 * np.exp(-1j * np.pi / 6)]),
]
n = np.arange(40)
circle = np.exp(1j * np.linspace(0, 2 * np.pi, 300))

fig, axes = plt.subplots(2, len(cases), figsize=(17, 6.5), gridspec_kw={"height_ratios": [1, 0.8]})
for column, (title, poles) in enumerate(cases):
    a = np.real(np.poly(poles))
    h = np.zeros(len(n))
    for k in range(len(n)):
        h[k] = (1.0 if k == 0 else 0.0) - sum(a[j] * h[k - j] for j in range(1, len(a)) if k - j >= 0)

    ax = axes[0, column]
    ax.plot(circle.real, circle.imag, "--", color="gray", linewidth=1)
    ax.plot(np.real(poles), np.imag(poles), "x", color="tab:red", markersize=12, markeredgewidth=2.5)
    ax.plot(0, 0, "o", markerfacecolor="none", color="tab:blue", markersize=10)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(title)

    ax = axes[1, column]
    ax.stem(n, h, basefmt=" ")
    ax.set_xlabel("n")
    ax.set_title("h[n]")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 $|H(z)|$ over the whole plane, and its trace on the unit circle

For the resonator with poles at $0.9e^{\pm i\pi/4}$ and zeros at $\pm1$, the left panel colours $\log_{10}|H(z)|$: bright peaks at the poles, dark pits at the zeros. The frequency response is what you see walking along the unit circle (white) from $\omega = 0$ to $\omega = \pi$: it passes near the pole at $\omega = \pi/4$ and through the zeros at the ends. The right panel plots exactly that walk.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

r, omega_0 = 0.9, np.pi / 4
poles = np.array([r * np.exp(1j * omega_0), r * np.exp(-1j * omega_0)])
zeros = np.array([1.0, -1.0])

grid = np.linspace(-1.6, 1.6, 600)
Z = grid[None, :] + 1j * grid[:, None]
H_plane = np.ones_like(Z)
for q in zeros:
    H_plane *= (Z - q)
for p in poles:
    H_plane /= (Z - p)
with np.errstate(divide="ignore"):
    log_gain = np.log10(np.abs(H_plane))

omega = np.linspace(0, np.pi, 1000)
on_circle = np.exp(1j * omega)
H_circle = np.prod([on_circle - q for q in zeros], axis=0) / np.prod([on_circle - p for p in poles], axis=0)

fig, (ax_plane, ax_response) = plt.subplots(1, 2, figsize=(14, 6))
image = ax_plane.imshow(np.clip(log_gain, -1.5, 1.5), extent=[-1.6, 1.6, -1.6, 1.6], origin="lower", cmap="magma")
fig.colorbar(image, ax=ax_plane, fraction=0.046, label="$\\log_{10}|H(z)|$")
ax_plane.plot(np.cos(np.linspace(0, 2 * np.pi, 300)), np.sin(np.linspace(0, 2 * np.pi, 300)), color="white", linewidth=1)
ax_plane.plot(on_circle.real, on_circle.imag, color="white", linewidth=3)
ax_plane.plot(poles.real, poles.imag, "x", color="cyan", markersize=12, markeredgewidth=2.5)
ax_plane.plot(zeros.real, zeros.imag, "o", markerfacecolor="none", color="cyan", markersize=11, markeredgewidth=2)
ax_plane.set_xlabel("Re")
ax_plane.set_ylabel("Im")
ax_plane.set_title("$\\log_{10}|H(z)|$, poles $0.9e^{\\pm i\\pi/4}$, zeros $\\pm 1$")

ax_response.plot(omega, np.abs(H_circle), color="tab:purple")
ax_response.axvline(omega_0, color="gray", linestyle="--", label="$\\omega_0 = \\pi/4$ (pole angle)")
ax_response.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi])
ax_response.set_xticklabels(["0", "$\\pi/4$", "$\\pi/2$", "$3\\pi/4$", "$\\pi$"])
ax_response.set_xlabel("$\\omega$ (radians per sample)")
ax_response.set_ylabel("$|H(e^{i\\omega})|$")
ax_response.set_title("Walking along the upper unit circle")
ax_response.grid(True, alpha=0.3)
ax_response.legend()
plt.tight_layout()
plt.show()

### 5.3 The gain is a ratio of distances

At one frequency, $\omega = 1.2$, draw the vectors from each zero and each pole to the point $e^{i\omega}$ on the circle. The gain is (product of green lengths) / (product of red lengths).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

r, omega_0 = 0.9, np.pi / 4
poles = [r * np.exp(1j * omega_0), r * np.exp(-1j * omega_0)]
zeros = [1.0 + 0j, -1.0 + 0j]
omega = 1.2
point = np.exp(1j * omega)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(np.cos(np.linspace(0, 2 * np.pi, 300)), np.sin(np.linspace(0, 2 * np.pi, 300)), "--", color="gray")
for q in zeros:
    ax.annotate("", xy=(point.real, point.imag), xytext=(q.real, q.imag), arrowprops=dict(arrowstyle="->", color="tab:green", linewidth=2))
    ax.plot(q.real, q.imag, "o", markerfacecolor="none", color="tab:blue", markersize=11, markeredgewidth=2)
    middle = (q + point) / 2
    ax.text(middle.real + 0.05, middle.imag - 0.08, f"{abs(point - q):.2f}", color="tab:green", fontsize=11)
for p in poles:
    ax.annotate("", xy=(point.real, point.imag), xytext=(p.real, p.imag), arrowprops=dict(arrowstyle="->", color="tab:red", linewidth=2))
    ax.plot(p.real, p.imag, "x", color="tab:red", markersize=12, markeredgewidth=2.5)
    middle = (p + point) / 2
    ax.text(middle.real + 0.05, middle.imag, f"{abs(point - p):.2f}", color="tab:red", fontsize=11)
ax.plot(point.real, point.imag, "ko", markersize=8)
ax.text(point.real - 0.05, point.imag + 0.1, "$e^{i\\omega}$, $\\omega = 1.2$", fontsize=12)

gain = np.prod([abs(point - q) for q in zeros]) / np.prod([abs(point - p) for p in poles])
ax.set_title(f"$|H(e^{{i\\omega}})|$ = (green product) / (red product) = {gain:.3f}")
ax.axhline(0, color="black", linewidth=0.6)
ax.axvline(0, color="black", linewidth=0.6)
ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.4, 1.4)
ax.set_aspect("equal")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
plt.show()

### 5.4 The notch filter at work

Top left: the notch's pole–zero plot (poles and zeros almost on top of each other at angle $\pm\pi/10$). Top right: its gain in hertz — flat except for a narrow dip at $50$ Hz, with sharper notches for $r$ closer to $1$. Bottom: the recorded signal with hum, and the filtered output against the true measurement; after a short start-up transient (the poles ring for roughly $1/(1 - r) = 20$ samples) the hum is gone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def notch(omega_0, r):
    b = np.array([1, -2 * np.cos(omega_0), 1])
    a = np.array([1, -2 * r * np.cos(omega_0), r**2])
    return b * a.sum() / b.sum(), a


def run_filter(b, a, x):
    y = np.zeros(len(x))
    for n in range(len(x)):
        total = sum(b[k] * x[n - k] for k in range(len(b)) if n - k >= 0)
        total -= sum(a[k] * y[n - k] for k in range(1, len(a)) if n - k >= 0)
        y[n] = total
    return y


sample_rate = 1000
omega_0 = 2 * np.pi * 50 / sample_rate
t = np.arange(0, 0.5, 1 / sample_rate)
measurement = np.sin(2 * np.pi * 3 * t) + 0.4 * np.sin(2 * np.pi * 11 * t)
recorded = measurement + np.sin(2 * np.pi * 50 * t + 0.3)
b, a = notch(omega_0, 0.95)
cleaned = run_filter(b, a, recorded)

fig = plt.figure(figsize=(14, 9))
ax_pz = fig.add_subplot(2, 2, 1)
ax_gain = fig.add_subplot(2, 2, 2)
ax_signal = fig.add_subplot(2, 1, 2)

circle = np.exp(1j * np.linspace(0, 2 * np.pi, 300))
ax_pz.plot(circle.real, circle.imag, "--", color="gray")
ax_pz.plot(np.roots(a).real, np.roots(a).imag, "x", color="tab:red", markersize=12, markeredgewidth=2.5, label="poles $0.95e^{\\pm i\\pi/10}$")
ax_pz.plot(np.roots(b).real, np.roots(b).imag, "o", markerfacecolor="none", color="tab:blue", markersize=12, markeredgewidth=2, label="zeros $e^{\\pm i\\pi/10}$")
ax_pz.axhline(0, color="black", linewidth=0.6)
ax_pz.axvline(0, color="black", linewidth=0.6)
ax_pz.set_aspect("equal")
ax_pz.set_xlim(-1.3, 1.3)
ax_pz.set_ylim(-1.3, 1.3)
ax_pz.set_xlabel("Re")
ax_pz.set_ylabel("Im")
ax_pz.set_title("Notch filter: pole-zero plot")
ax_pz.legend(loc="lower left", fontsize=9)

frequencies = np.linspace(0, 150, 3000)
omega = 2 * np.pi * frequencies / sample_rate
for r in [0.8, 0.95, 0.99]:
    bb, aa = notch(omega_0, r)
    e = np.exp(-1j * omega)
    H = (bb[0] + bb[1] * e + bb[2] * e**2) / (aa[0] + aa[1] * e + aa[2] * e**2)
    ax_gain.plot(frequencies, np.abs(H), label=f"r = {r}")
ax_gain.set_xlabel("frequency (Hz)")
ax_gain.set_ylabel("$|H|$")
ax_gain.set_title("Gain of the notch for three pole radii")
ax_gain.grid(True, alpha=0.3)
ax_gain.legend()

ax_signal.plot(t * 1000, recorded, color="lightgray", label="recorded (with 50 Hz hum)")
ax_signal.plot(t * 1000, measurement, color="tab:blue", linewidth=3, alpha=0.4, label="true measurement")
ax_signal.plot(t * 1000, cleaned, color="tab:orange", label="notch-filtered (r = 0.95)")
ax_signal.set_xlabel("t (ms)")
ax_signal.set_title("Removing hum by placing a zero on the unit circle")
ax_signal.grid(True, alpha=0.3)
ax_signal.legend(loc="upper right", ncol=3, fontsize=9)
ax_signal.set_ylim(-2.6, 3.0)
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Find the poles of $y[n] = y[n-1] - 0.5\,y[n-2] + x[n]$. Is it stable? Describe its impulse response (ratio per sample and period).
2. Write the $z$-transform of the finite sequence $x = (1, 2, 3)$ and find its zeros.
3. For the differencer $y[n] = x[n] - x[n-1]$, find $H(z)$, its zero, and $|H(e^{i\omega})|$. Low-pass or high-pass?

### Intermediate

4. **Draw it first, then compute.** $H(z) = \dfrac{1}{1 + 0.8z^{-1}}$. Mark the pole on a sketch of the unit circle. Before computing anything, predict: is the gain larger at $\omega = 0$ or $\omega = \pi$? What does the impulse response look like? Then compute $|H|$ at both frequencies and $h[n]$.
5. Find the impulse response of $y[n] = 0.8\,y[n-1] - 0.64\,y[n-2] + x[n]$ in closed form. (Hint: identify $r$ and $\theta$.)
6. Design an FIR filter with zeros at $z = 1$ and $z = -1$ only (plus poles at the origin). Write the difference equation, normalise it so that the peak gain is $1$, and find where the peak is.

### Advanced

7. Show that $\sum_n |h[n]| < \infty$ implies BIBO stability, and that for $h[n] = \sum_j r_j p_j^{\,n}$ (distinct poles, all $r_j \ne 0$) it holds if and only if every $|p_j| < 1$.
8. For the notch filter, explain geometrically why $|H(e^{i\omega})| \to 1$ for every $\omega \ne \pm\omega_0$ as $r \to 1$. Then measure numerically the width of the notch (the frequency band where $|H| < 1/\sqrt2$) for $r = 0.9, 0.95, 0.99$ and compare with $2(1 - r)$.
9. Show that for a stable filter the response to the unit step $x[n] = 1$ ($n \ge 0$) tends to $H(1)$. Verify for Example 1.

## 7. Solutions

1. Poles: roots of $z^2 - z + 0.5$, $z = \frac{1 \pm i}{2} = \frac{1}{\sqrt2}e^{\pm i\pi/4}$. $|p| = 0.707 < 1$: stable. The impulse response shrinks by $0.707$ per sample and turns $\pi/4$ per sample: period $8$ samples.
2. $X(z) = 1 + 2z^{-1} + 3z^{-2} = \dfrac{z^2 + 2z + 3}{z^2}$. Zeros at $z = -1 \pm i\sqrt2$ (outside the unit circle; harmless, zeros do not affect stability), double pole at $0$.
3. $H(z) = 1 - z^{-1} = \dfrac{z - 1}{z}$, zero at $z = 1$ (DC). $|H(e^{i\omega})| = |1 - e^{-i\omega}| = |e^{-i\omega/2}|\,|e^{i\omega/2} - e^{-i\omega/2}| = 2\left|\sin\frac{\omega}{2}\right|$: $0$ at DC, $2$ at Nyquist — high-pass.
4. Sketch: the pole is at $-0.8$, close to the point $-1$ where $\omega = \pi$. So the gain should be large at $\omega = \pi$ and small at $\omega = 0$ (high-pass), and the impulse response should alternate in sign. Computation: $|H(1)| = \frac{1}{1.8} = 0.556$, $|H(-1)| = \frac{1}{0.2} = 5$, and $h[n] = (-0.8)^n$.
5. $2r\cos\theta = 0.8$ and $r^2 = 0.64$ give $r = 0.8$, $\cos\theta = 0.5$, $\theta = \pi/3$. By Example 3, $h[n] = 0.8^n\dfrac{\sin((n+1)\pi/3)}{\sin(\pi/3)}$.
6. $H(z) = g(1 - z^{-1})(1 + z^{-1}) = g(1 - z^{-2})$, so $y[n] = g(x[n] - x[n-2])$. $|1 - e^{-2i\omega}| = 2|\sin\omega|$, maximal at $\omega = \pi/2$ (a quarter of the sampling rate). With $g = \tfrac12$ the peak gain is $1$: a crude band-pass.
7. If $|x[n]| \le M$ then $|y[n]| = \left|\sum_k h[k]x[n-k]\right| \le M\sum_k|h[k]|$. For geometric sums: if all $|p_j| < 1$, $\sum_n|h[n]| \le \sum_j|r_j|\frac{1}{1 - |p_j|} < \infty$. Conversely, if some $|p_j| \ge 1$, let $\rho = \max|p_j| \ge 1$; the terms with $|p_j| = \rho$ form $\rho^n$ times a nonzero almost-periodic sequence, which does not tend to $0$, so $h[n] \not\to 0$ and the sum diverges.
8. For a point $e^{i\omega}$ away from $e^{\pm i\omega_0}$, the distance to the zero $e^{i\omega_0}$ and to the pole $re^{i\omega_0}$ differ by at most $1 - r$, so their ratio tends to $1$ as $r \to 1$, and the same for the conjugate pair; the normalisation $g \to 1$ too. Only very near $\omega_0$ is the zero much closer than the pole. Numerically the widths are about $0.18$, $0.098$ and $0.020$ rad per sample, close to $2(1 - r) = 0.2, 0.1, 0.02$ and closer the nearer $r$ is to $1$.
9. The step is $e^{i\omega n}$ with $\omega = 0$. Writing $y[n] = \sum_{k=0}^{n}h[k] \to \sum_{k \ge 0} h[k] = H(1)$, which converges because the filter is stable. For Example 1, $H(1) = \frac{1}{1 - 0.5} = 2$, and $y[n] = \sum_{k \le n}0.5^k = 2 - 0.5^n \to 2$.

Numerical checks:

In [ ]:
import numpy as np


def run_filter(b, a, x):
    y = np.zeros(len(x))
    for n in range(len(x)):
        total = sum(b[k] * x[n - k] for k in range(len(b)) if n - k >= 0)
        total -= sum(a[k] * y[n - k] for k in range(1, len(a)) if n - k >= 0)
        y[n] = total
    return y


def gain(b, a, omega):
    e = np.exp(-1j * np.asarray(omega))
    return np.abs(sum(bk * e**k for k, bk in enumerate(b)) / sum(ak * e**k for k, ak in enumerate(a)))


impulse = np.zeros(20)
impulse[0] = 1.0
print("Ex 1 poles:", np.roots([1, -1, 0.5]), " |p| =", np.abs(np.roots([1, -1, 0.5])))
print("Ex 2 zeros:", np.roots([1, 2, 3]))
print("Ex 3 |H| at pi/3:", gain([1, -1], [1], np.pi / 3), " 2 sin(pi/6) =", 2 * np.sin(np.pi / 6))
print("Ex 4 gains:", gain([1], [1, 0.8], [0, np.pi]), " h =", np.round(run_filter([1], [1, 0.8], impulse)[:5], 4))
n = np.arange(20)
print("Ex 5:", np.allclose(run_filter([1], [1, -0.8, 0.64], impulse), 0.8**n * np.sin((n + 1) * np.pi / 3) / np.sin(np.pi / 3)))
omega = np.linspace(0, np.pi, 100001)
g6 = gain([0.5, 0, -0.5], [1], omega)
print("Ex 6: peak gain", round(g6.max(), 6), "at omega =", round(omega[np.argmax(g6)], 5), "(pi/2 =", round(np.pi / 2, 5), ")")

omega_0 = np.pi / 10
for r in [0.9, 0.95, 0.99]:
    b = np.array([1, -2 * np.cos(omega_0), 1])
    a = np.array([1, -2 * r * np.cos(omega_0), r**2])
    b = b * a.sum() / b.sum()
    below = omega[gain(b, a, omega) < 1 / np.sqrt(2)]
    print(f"Ex 8: r = {r}: notch width {below.max() - below.min():.4f} rad/sample, 2(1 - r) = {2 * (1 - r):.4f}")

step = np.ones(60)
print("Ex 9: step response of Ex 1 ends at", round(run_filter([1], [1, -0.5], step)[-1], 10), " H(1) =", gain([1], [1, -0.5], 0))

## 8. Conclusion

- A digital LTI filter is a difference equation; its $z$-transform $X(z) = \sum x[n]z^{-n}$ is a Laurent series in which $z^{-1}$ means "delay by one sample".
- The **transfer function** $H(z) = B(z^{-1})/A(z^{-1})$ is rational; its **poles** (roots of $A$) and **zeros** (roots of $B$) come in conjugate pairs for real filters.
- Partial fractions write the impulse response as $\sum r_j p_j^{\,n}$: each pole is a geometric sequence, a complex pair is a damped oscillation. **Stable $\iff$ all poles inside the unit circle** — the discrete twin of "left half-plane" from notebook 02.
- On the unit circle, $H(e^{i\omega})$ is the gain and phase at frequency $\omega$; its modulus is a ratio of distances to zeros and poles, so poles near the circle make peaks and zeros on the circle make notches.
- Placing zeros at $e^{\pm i\omega_0}$ and poles just inside gives a notch filter; the recurrence alone, a dozen lines of Python, removed the 50 Hz hum.

**Next:** notebook 05 leaves linear systems behind. Iterating the simplest *nonlinear* complex map, $z \mapsto z^2 + c$, produces the Mandelbrot set and Julia sets — where the question "does the orbit stay bounded?" has an infinitely intricate answer.